# H3 — Spaces-as-Compiler: Context Density Curve Analysis

This notebook analyzes the relationship between Copilot Space context density (tokens) and cloud-agent output quality.

**Hypothesis H₁**: There is a measurable monotonic relationship (r² ≥ 0.3) between context density and agent quality, with a locatable inflection point (diminishing returns knee).

**Falsified if**: No monotonic trend (r² < 0.3), i.e., context is noise.

In [ ]:
import json
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit
from typing import Optional

print('Dependencies loaded successfully.')

## 1. Load Raw Trial Data

In [ ]:
RAW_DIR = pathlib.Path('h3-spaces-density/analysis/raw')

# Context token counts for each space level
SPACE_TOKENS = {
    '0k': 0,
    '2k': 2_000,
    '8k': 8_000,
    '32k': 32_000,
    '128k': 128_000,
    'max': None,  # Will be filled from data if available
}

all_trials = []

for json_file in sorted(RAW_DIR.glob('*.json')):
    data = json.loads(json_file.read_text())
    if isinstance(data, list):
        all_trials.extend(data)
    elif isinstance(data, dict):
        all_trials.append(data)

print(f'Loaded {len(all_trials)} trial records from {len(list(RAW_DIR.glob("*.json")))} files.')

# Show sample
if all_trials:
    print('\nSample record:')
    print(json.dumps(all_trials[0], indent=2))
else:
    print('\nNo data yet. Run the h3-density-sweep.yml workflow to collect data.')

## 2. Aggregate by Space Level

In [ ]:
from collections import defaultdict

space_data = defaultdict(list)

for trial in all_trials:
    space = trial.get('space', 'unknown')
    metrics = trial.get('metrics', {})
    composite = metrics.get('composite_quality')
    
    # Compute composite if not already done
    if composite is None:
        tp = metrics.get('tests_pass')
        lc = metrics.get('lint_clean')
        ra = metrics.get('review_approval')
        hs = metrics.get('human_score')
        
        if all(v is not None for v in [tp, lc, ra, hs]):
            composite = 0.3 * tp + 0.2 * lc + 0.2 * ra + 0.3 * (hs / 5.0)
    
    if composite is not None:
        space_data[space].append(composite)

print('Trials with composite quality scores per space:')
for space in ['0k', '2k', '8k', '32k', '128k', 'max']:
    n = len(space_data.get(space, []))
    avg = np.mean(space_data[space]) if space_data.get(space) else None
    tokens = SPACE_TOKENS.get(space, '?')
    avg_str = f"{avg:.3f}" if avg is not None else "—"
    print(f'  {space:6s} ({tokens:>7} tokens): n={n}, avg_quality={avg_str}')

## 3. Prepare Data for Curve Fitting

In [ ]:
# Collect (tokens, quality) pairs for spaces that have data
x_tokens = []
y_quality = []
x_labels = []

for space in ['0k', '2k', '8k', '32k', '128k', 'max']:
    scores = space_data.get(space, [])
    if not scores:
        continue
    
    token_count = SPACE_TOKENS.get(space)
    if token_count is None:
        # Try to get from trial data
        for trial in all_trials:
            if trial.get('space') == space and trial.get('actual_token_count'):
                token_count = trial['actual_token_count']
                break
        if token_count is None:
            print(f'  Skipping {space}: no token count available')
            continue
    
    for score in scores:
        x_tokens.append(token_count)
        y_quality.append(score)
        x_labels.append(space)

x = np.array(x_tokens, dtype=float)
y = np.array(y_quality, dtype=float)

print(f'Data points ready for analysis: n={len(x)}')
if len(x) > 0:
    print(f'Token range: [{x.min():.0f}, {x.max():.0f}]')
    print(f'Quality range: [{y.min():.3f}, {y.max():.3f}]')

## 4. Statistical Analysis: Pearson r² and Spearman ρ

In [ ]:
if len(x) < 4:
    print('Insufficient data for statistical analysis. Need at least 4 data points.')
    print(f'Current: {len(x)} points. Collect more trials before running this analysis.')
else:
    # Pearson correlation
    pearson_r, pearson_p = stats.pearsonr(x, y)
    pearson_r2 = pearson_r ** 2
    
    # Spearman rank correlation
    spearman_rho, spearman_p = stats.spearmanr(x, y)
    
    print('=== Statistical Analysis ===')
    print(f'Pearson r:      {pearson_r:.4f}')
    print(f'Pearson r²:     {pearson_r2:.4f}  (threshold: ≥ 0.30)')
    print(f'Pearson p-val:  {pearson_p:.4f}')
    print(f'Spearman ρ:     {spearman_rho:.4f}')
    print(f'Spearman p-val: {spearman_p:.4f}')
    print()
    
    # Verdict
    alpha = 0.05 / 3  # Bonferroni correction for 3 hypotheses
    if pearson_r2 >= 0.30 and pearson_p < alpha:
        print('✅ H₁ SUPPORTED: r² ≥ 0.30 and statistically significant')
    elif pearson_r2 >= 0.30:
        print('⚠️  H₁ PARTIALLY SUPPORTED: r² ≥ 0.30 but not significant at Bonferroni-corrected α')
    else:
        print('❌ H₁ NOT SUPPORTED: r² < 0.30 — context density does not predict quality')

## 5. Log Curve Fitting

In [ ]:
def log_curve(x, a, b):
    """Logarithmic model: y = a * log(x + 1) + b"""
    return a * np.log(x + 1) + b

def power_curve(x, a, b, c):
    """Power law model: y = a * x^b + c"""
    return a * np.power(x + 1, b) + c

fitted_log = None
fitted_power = None

if len(x) >= 4:
    try:
        params_log, _ = curve_fit(log_curve, x, y, p0=[0.1, 0.1], maxfev=10000)
        y_pred_log = log_curve(x, *params_log)
        ss_res = np.sum((y - y_pred_log) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2_log = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        fitted_log = {'params': params_log, 'r2': r2_log}
        print(f'Log curve fit: a={params_log[0]:.4f}, b={params_log[1]:.4f}, r²={r2_log:.4f}')
    except Exception as e:
        print(f'Log curve fit failed: {e}')
    
    try:
        params_power, _ = curve_fit(power_curve, x, y, p0=[0.1, 0.1, 0.1], maxfev=10000)
        y_pred_power = power_curve(x, *params_power)
        ss_res = np.sum((y - y_pred_power) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2_power = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        fitted_power = {'params': params_power, 'r2': r2_power}
        print(f'Power curve fit: a={params_power[0]:.4f}, b={params_power[1]:.4f}, c={params_power[2]:.4f}, r²={r2_power:.4f}')
    except Exception as e:
        print(f'Power curve fit failed: {e}')
else:
    print('Insufficient data for curve fitting.')

## 6. Knee Point Detection (Second-Derivative Method)

In [ ]:
knee_token = None

if fitted_log is not None:
    # Evaluate log curve on a dense grid
    x_dense = np.linspace(1, max(x.max(), 1), 10000)
    a, b = fitted_log['params']
    y_dense = log_curve(x_dense, a, b)
    
    # First derivative (slope)
    dy = np.gradient(y_dense, x_dense)
    
    # Second derivative (rate of change of slope)
    d2y = np.gradient(dy, x_dense)
    
    # Knee = point of maximum curvature (most negative second derivative for concave-up curves)
    knee_idx = np.argmin(d2y)
    knee_token = x_dense[knee_idx]
    knee_quality = y_dense[knee_idx]
    
    print(f'=== Knee Point Detection ===')
    print(f'Inflection point (knee): {knee_token:,.0f} tokens')
    print(f'Quality at knee:         {knee_quality:.3f}')
    print()
    
    # Classify the knee relative to our space levels
    for space, tokens in [('0k', 0), ('2k', 2000), ('8k', 8000), 
                           ('32k', 32000), ('128k', 128000)]:
        next_tokens = [0, 2000, 8000, 32000, 128000, float('inf')]
        i = [0, 2000, 8000, 32000, 128000].index(tokens)
        if tokens <= knee_token < next_tokens[i + 1]:
            print(f'Knee falls in the {space} → next space range.')
            break
else:
    print('Knee detection requires a successful log curve fit.')

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Raw data + fitted curves ---
ax = axes[0]

if len(x) > 0:
    # Scatter plot of individual trials
    ax.scatter(x, y, alpha=0.6, color='steelblue', zorder=5, label='Individual trials')
    
    # Space-level means
    for space in ['0k', '2k', '8k', '32k', '128k', 'max']:
        scores = space_data.get(space, [])
        tk = SPACE_TOKENS.get(space)
        if scores and tk is not None:
            ax.scatter([tk], [np.mean(scores)], s=200, zorder=10, 
                      color='darkblue', marker='D', label=f'{space} mean' if space == '0k' else '_')
            ax.annotate(space, (tk, np.mean(scores)), textcoords='offset points',
                       xytext=(5, 5), fontsize=9)
    
    # Fitted curves
    x_plot = np.linspace(0, max(x.max() * 1.1, 1000), 500)
    
    if fitted_log:
        y_log = log_curve(x_plot, *fitted_log['params'])
        ax.plot(x_plot, y_log, 'r-', label=f'Log fit (r²={fitted_log["r2"]:.3f})', linewidth=2)
    
    # Knee point
    if knee_token:
        ax.axvline(knee_token, color='orange', linestyle='--', label=f'Knee ({knee_token:,.0f} tokens)')
    
    ax.set_xlabel('Context Tokens')
    ax.set_ylabel('Composite Quality Score')
    ax.set_title('H3: Context Density vs Agent Quality')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

else:
    ax.text(0.5, 0.5, 'No data yet\nRun h3-density-sweep.yml to collect data',
            ha='center', va='center', transform=ax.transAxes, fontsize=14, color='gray')
    ax.set_title('H3: Context Density vs Agent Quality (No Data)')

# --- Plot 2: Quality by space (box plot) ---
ax2 = axes[1]

spaces_with_data = [s for s in ['0k', '2k', '8k', '32k', '128k', 'max'] if space_data.get(s)]

if spaces_with_data:
    box_data = [space_data[s] for s in spaces_with_data]
    bp = ax2.boxplot(box_data, labels=spaces_with_data, patch_artist=True)
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(spaces_with_data)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    ax2.set_xlabel('Space Context Level')
    ax2.set_ylabel('Composite Quality Score')
    ax2.set_title('Quality Distribution by Space Level')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_ylim(0, 1.05)
else:
    ax2.text(0.5, 0.5, 'No data yet', ha='center', va='center', 
             transform=ax2.transAxes, fontsize=14, color='gray')
    ax2.set_title('Quality Distribution by Space Level (No Data)')

plt.tight_layout()
plt.savefig('h3-spaces-density/analysis/density_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to h3-spaces-density/analysis/density_curve.png')

## 8. Final Verdict

In [ ]:
print('=' * 60)
print('H3 VERDICT SUMMARY')
print('=' * 60)
print()

total_trials = len(x)
print(f'Total trials with quality scores: {total_trials} / 30')

if total_trials < 30:
    print(f'\n⏳ PENDING — Need {30 - total_trials} more trials before final verdict.')
elif len(x) < 4:
    print('\n⏳ PENDING — Insufficient scored trials for statistical analysis.')
else:
    alpha_bonferroni = 0.05 / 3
    
    monotonic = spearman_rho > 0
    significant = pearson_p < alpha_bonferroni
    strong = pearson_r2 >= 0.30
    knee_found = knee_token is not None
    
    print(f'Pearson r²:     {pearson_r2:.4f}  (threshold ≥ 0.30: {"✅" if strong else "❌"})')
    print(f'Monotonic trend:{" ✅" if monotonic else " ❌"} (Spearman ρ = {spearman_rho:.4f})')
    print(f'Statistical sig:{" ✅" if significant else " ❌"} (p = {pearson_p:.4f}, α_adj = {alpha_bonferroni:.4f})')
    print(f'Knee found:     {" ✅" if knee_found else " ❌"}', end='')
    if knee_found:
        print(f' at {knee_token:,.0f} tokens')
    else:
        print()
    print()
    
    if strong and monotonic and significant:
        print('✅ H₁ CONFIRMED: Monotonic relationship exists (r² ≥ 0.30, significant).')
        if knee_found:
            print(f'   Inflection point located at ~{knee_token:,.0f} tokens.')
            print('   Spaces Optimizer SaaS hypothesis is viable.')
    elif strong and monotonic:
        print('⚠️  H₁ PARTIALLY CONFIRMED: Trend exists but not significant at Bonferroni α.')
        print('   Consider extending to more trials for significance.')
    else:
        print('❌ H₁ FALSIFIED (H₀ not rejected): r² < 0.30 — context density does not')
        print('   reliably predict agent quality for this task type.')

print()
print('=' * 60)